<a href="https://colab.research.google.com/github/anindyabera3/graph_analysis/blob/main/ibm_amlsim_graph_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#pip install kaggle

In [ ]:
from google.colab import files
files.upload()   # upload kaggle.json

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"anindyabera3","key":"f80df47f08bf3fd8975bd4b4e4be3e2a"}'}

In [ ]:
!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
!kaggle datasets list | head

ref                                                           title                                                     size  lastUpdated                 downloadCount  voteCount  usabilityRating  
------------------------------------------------------------  --------------------------------------------------  ----------  --------------------------  -------------  ---------  ---------------  
wardabilal/spotify-global-music-dataset-20092025              Spotify Global Music Dataset (2009–2025)               1289021  2025-11-11 09:43:05.933000          10643        241  1.0              
rohiteng/amazon-sales-dataset                                 Amazon Sales Dataset                                   4037578  2025-11-23 14:29:37.973000           3279         46  1.0              
khushikyad001/ai-impact-on-jobs-2030                          AI Impact on Jobs 2030                                   87410  2025-11-09 17:58:05.410000           6186        141  1.0              
mayabennet

In [ ]:
!kaggle datasets download -d anshankul/ibm-amlsim-example-dataset
!unzip -o ibm-amlsim-example-dataset.zip

Dataset URL: https://www.kaggle.com/datasets/anshankul/ibm-amlsim-example-dataset
License(s): other
  0% 0.00/11.4M [00:00<?, ?B/s]
100% 11.4M/11.4M [00:00<00:00, 1.21GB/s]
Archive:  ibm-amlsim-example-dataset.zip
  inflating: accounts.csv            
  inflating: alerts.csv              
  inflating: transactions.csv        


In [ ]:
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import community as community_louvain
import community.community_louvain as community_louvain

df_accounts = pd.read_csv("accounts.csv")
df_transactions = pd.read_csv("transactions.csv")

In [ ]:
df_accounts.head()

,ACCOUNT_ID,CUSTOMER_ID,INIT_BALANCE,COUNTRY,ACCOUNT_TYPE,IS_FRAUD,TX_BEHAVIOR_ID
0,0,C_0,184.44,US,I,False,1
1,1,C_1,175.80,US,I,False,1
2,2,C_2,142.06,US,I,False,1
3,3,C_3,125.89,US,I,False,1
4,4,C_4,151.13,US,I,False,1


In [ ]:
df_transactions.head()

,TX_ID,SENDER_ACCOUNT_ID,RECEIVER_ACCOUNT_ID,TX_TYPE,TX_AMOUNT,TIMESTAMP,IS_FRAUD,ALERT_ID
0,1,6456,9069,TRANSFER,465.05,0,False,-1
1,2,7516,9543,TRANSFER,564.64,0,False,-1
2,3,2445,9356,TRANSFER,598.94,0,False,-1
3,4,2576,4617,TRANSFER,466.07,0,False,-1
4,5,3524,1773,TRANSFER,405.63,0,False,-1


In [ ]:
G = nx.MultiDiGraph()

In [ ]:
# Add nodes
for _, row in df_accounts.iterrows():
    G.add_node(
        row['ACCOUNT_ID'],
        customer=row.get('CUSTOMER_ID'),
        type=row.get('ACCOUNT_TYPE'),
        balance=row.get('INIT_BALANCE')
    )

# Add edges (MULTIPLE allowed)
for _, row in df_transactions.iterrows():
    G.add_edge(
        row['SENDER_ACCOUNT_ID'],
        row['RECEIVER_ACCOUNT_ID'],
        amount=row['TX_AMOUNT'],
        timestamp=row['TIMESTAMP'],
        txn_id=row.get('TX_ID')
    )


In [ ]:
multi_pairs = [
    (u, v, G.number_of_edges(u, v))
    for u, v in G.edges()
    if G.number_of_edges(u, v) > 5
]

multi_pairs[:10]


[(1, 884, 24),
 (1, 884, 24),
 (1, 884, 24),
 (1, 884, 24),
 (1, 884, 24),
 (1, 884, 24),
 (1, 884, 24),
 (1, 884, 24),
 (1, 884, 24),
 (1, 884, 24)]

In [ ]:
multi_pairs_comm1 = [
    (u, v, G_comm1.number_of_edges(u, v))
    for u, v in G_comm1.edges()
    if G_comm1.number_of_edges(u, v) > 5
]

In [ ]:
G_multi = G.edge_subgraph([(u, v, k)
                           for u, v, k in G.edges(keys=True)
                           if G.number_of_edges(u, v) > 1]).copy()

In [ ]:
# Create subgraph of repeated-transaction edges

#for u, v, count in multi_pairs:
#    for k, data in G.get_edge_data(u, v).items():  # keep original edges
#        G.add_edge(u, v, **data)

print("Subgraph size:", G_multi.number_of_nodes(), "nodes,", G_multi.number_of_edges(), "edges")

Subgraph size: 9999 nodes, 1314775 edges


In [ ]:
plt.figure(figsize=(14, 12))

# Layout
pos = nx.spring_layout(G_multi, k=0.35, iterations=50)

# --- Node sizes: number of repeated payments received ---
node_size = [
    400 + 80 * G_multi.in_degree(n)
    for n in G_multi.nodes()
]

# --- Edge widths: count multi-edges between each pair ---
edge_widths = [
    G_multi.number_of_edges(u, v)
    for u, v in G_multi.edges()
]

# Draw nodes
nx.draw_networkx_nodes(
    G_multi, pos,
    node_color='orange',
    node_size=node_size,
    alpha=0.85
)

# Draw edges
nx.draw_networkx_edges(
    G_multi, pos,
    width=edge_widths,
    alpha=0.6,
    arrows=True,
    arrowstyle='-|>',
    arrowsize=15
)

# Labels
nx.draw_networkx_labels(G_multi, pos, font_size=9)

plt.title("2D Repeated-Transaction Network (Multi-Edge Hotspots)", fontsize=18)
plt.axis("off")
plt.show()


KeyboardInterrupt: 

Error in callback <function _draw_all_if_interactive at 0x7d9d80cd5ee0> (for post_execute):


KeyboardInterrupt: 

Error in callback <function flush_figures at 0x7d9d80cd5940> (for post_execute):


KeyboardInterrupt: 

In [ ]:
G_weighted = nx.DiGraph()

for u, v, data in G.edges(data=True):
    amt = data['amount']
    if G_weighted.has_edge(u, v):
        G_weighted[u][v]['weight'] += amt
    else:
        G_weighted.add_edge(u, v, weight=amt)


In [ ]:
centrality = nx.betweenness_centrality(G_weighted, weight='weight')

KeyboardInterrupt: 

In [ ]:
G = nx.DiGraph()

# Add accounts as nodes
for _, row in df_accounts.iterrows():
    G.add_node(
        row['ACCOUNT_ID'],
        customer=row.get('CUSTOMER_ID'),
        type=row.get('ACCOUNT_TYPE'),
        balance=row.get('INIT_BALANCE'),
        is_sar=row.get('IS_FRAUD')
    )

# Add transactions as directed edges
for _, row in df_transactions.iterrows():
    G.add_edge(
        row['SENDER_ACCOUNT_ID'],
        row['RECEIVER_ACCOUNT_ID'],
        amount=row['TX_AMOUNT'],
        timestamp=row['TIMESTAMP']
    )

# %% [code] {"execution":{"iopub.status.busy":"2025-12-07T14:09:07.458469Z","iopub.execute_input":"2025-12-07T14:09:07.458779Z","iopub.status.idle":"2025-12-07T14:09:07.468633Z","shell.execute_reply.started":"2025-12-07T14:09:07.458756Z","shell.execute_reply":"2025-12-07T14:09:07.467577Z"}}
print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())
print("Directed:", G.is_directed())

# %% [code] {"execution":{"iopub.status.busy":"2025-12-07T14:09:20.852699Z","iopub.execute_input":"2025-12-07T14:09:20.853041Z","iopub.status.idle":"2025-12-07T14:09:31.429782Z","shell.execute_reply.started":"2025-12-07T14:09:20.853018Z","shell.execute_reply":"2025-12-07T14:09:31.427912Z"}}
partition = community_louvain.best_partition(G.to_undirected())
nx.set_node_attributes(G, partition, "ring")

# %% [code] {"execution":{"iopub.status.busy":"2025-12-07T14:09:31.431966Z","iopub.execute_input":"2025-12-07T14:09:31.432324Z","iopub.status.idle":"2025-12-07T14:09:31.469608Z","shell.execute_reply.started":"2025-12-07T14:09:31.432295Z","shell.execute_reply":"2025-12-07T14:09:31.468786Z"}}
ring_df = pd.DataFrame.from_dict(partition, orient='index', columns=['ring'])
top_ring = ring_df['ring'].value_counts().idxmax()
ring_nodes = ring_df[ring_df['ring'] == top_ring].index.tolist()

fraud_ring = G.subgraph(ring_nodes)

# %% [code] {"execution":{"iopub.status.busy":"2025-12-07T13:42:53.872918Z","iopub.execute_input":"2025-12-07T13:42:53.873282Z","iopub.status.idle":"2025-12-07T13:43:33.005743Z","shell.execute_reply.started":"2025-12-07T13:42:53.873263Z","shell.execute_reply":"2025-12-07T13:43:33.004251Z"}}
plt.figure(figsize=(10,10))
pos = nx.spring_layout(fraud_ring, k=0.25)

nx.draw_networkx_nodes(fraud_ring, pos, node_size=300, alpha=0.9, node_color='orange')
nx.draw_networkx_edges(fraud_ring, pos, arrows=True, alpha=0.4)
nx.draw_networkx_labels(fraud_ring, pos, font_size=8)

plt.title("Largest Fraud Community")
plt.axis('off')
plt.show()

# %% [code] {"execution":{"iopub.status.busy":"2025-12-07T14:15:45.583382Z","iopub.execute_input":"2025-12-07T14:15:45.583650Z","iopub.status.idle":"2025-12-07T14:15:45.591598Z","shell.execute_reply.started":"2025-12-07T14:15:45.583636Z","shell.execute_reply":"2025-12-07T14:15:45.589356Z"}}
community_id = 1 # pick the community you want

# All nodes belonging to that community
nodes_in_comm = [n for n, c in partition.items() if c == community_id]

print("Nodes in this community:", len(nodes_in_comm))


# %% [code] {"execution":{"iopub.status.busy":"2025-12-07T14:16:00.251705Z","iopub.execute_input":"2025-12-07T14:16:00.251996Z","iopub.status.idle":"2025-12-07T14:16:00.262713Z","shell.execute_reply.started":"2025-12-07T14:16:00.251974Z","shell.execute_reply":"2025-12-07T14:16:00.261568Z"}}
subG = G.subgraph(nodes_in_comm).copy()
print("Subgraph size:", subG.number_of_nodes(), "nodes,", subG.number_of_edges(), "edges")


# %% [code] {"execution":{"iopub.status.busy":"2025-12-07T14:16:49.304011Z","iopub.execute_input":"2025-12-07T14:16:49.304726Z","iopub.status.idle":"2025-12-07T14:16:50.572832Z","shell.execute_reply.started":"2025-12-07T14:16:49.304705Z","shell.execute_reply":"2025-12-07T14:16:50.571607Z"}}
import matplotlib.pyplot as plt
import networkx as nx

pos = nx.spring_layout(subG, seed=42)

plt.figure(figsize=(10, 8))
nx.draw(
    subG,
    pos,
    node_size=120,
    node_color="orange",
    edge_color="gray",
    alpha=0.8,
    linewidths=1,
)
plt.title(f"Community {community_id} — Local Fraud Ring", fontsize=14)
plt.show()


# %% [code] {"execution":{"iopub.status.busy":"2025-12-07T14:17:23.123219Z","iopub.execute_input":"2025-12-07T14:17:23.123623Z","iopub.status.idle":"2025-12-07T14:17:24.181269Z","shell.execute_reply.started":"2025-12-07T14:17:23.123597Z","shell.execute_reply":"2025-12-07T14:17:24.179947Z"}}
edge_colors = [subG[u][v]['amount'] for u,v in subG.edges()]

plt.figure(figsize=(11, 9))
nx.draw(
    subG,
    pos,
    node_color="skyblue",
    edge_color=edge_colors,
    edge_cmap=plt.cm.plasma,
    node_size=140,
    width=2,
)
plt.title(f"Community {community_id} — Amount-Colored Edges", fontsize=14)
plt.show()

# %% [code] {"execution":{"iopub.status.busy":"2025-12-07T14:19:13.377925Z","iopub.execute_input":"2025-12-07T14:19:13.378272Z","iopub.status.idle":"2025-12-07T14:19:14.626431Z","shell.execute_reply.started":"2025-12-07T14:19:13.378256Z","shell.execute_reply":"2025-12-07T14:19:14.625362Z"}}
import numpy as np

btw = nx.betweenness_centrality(subG, normalized=True)

node_sizes = [3000 * btw[n] for n in subG.nodes()]  # scale up
node_colors = [btw[n] for n in subG.nodes()]

plt.figure(figsize=(12, 10))
nx.draw(
    subG,
    pos,
    node_size=node_sizes,
    node_color=node_colors,
    cmap=plt.cm.plasma,
    edge_color="gray",
    alpha=0.75
)

plt.title(f"Community {community_id} — Mule Candidates (Betweenness Score)", fontsize=14)
plt.colorbar(plt.cm.ScalarMappable(cmap="plasma"))
plt.show()


# %% [code] {"execution":{"iopub.status.busy":"2025-12-07T14:55:33.201534Z","iopub.execute_input":"2025-12-07T14:55:33.202338Z","iopub.status.idle":"2025-12-07T14:55:33.260316Z","shell.execute_reply.started":"2025-12-07T14:55:33.202242Z","shell.execute_reply":"2025-12-07T14:55:33.259396Z"}}
THRESHOLD = 1000000

# Filter only transactions where BOTH source & target are in Community 1
tx_comm1 = df_transactions[
    df_transactions["SENDER_ACCOUNT_ID"].isin(nodes_in_comm) &
    df_transactions["RECEIVER_ACCOUNT_ID"].isin(nodes_in_comm)
]

# Now extract only high-value ones
high_value_tx_comm1 = tx_comm1[tx_comm1["TX_AMOUNT"] > THRESHOLD]

print("High-value transactions in Community 1:", len(high_value_tx_comm1))

out_hv = high_value_tx_comm1.groupby("SENDER_ACCOUNT_ID").size().rename("hv_out")
in_hv  = high_value_tx_comm1.groupby("RECEIVER_ACCOUNT_ID").size().rename("hv_in")

hv_df = pd.concat([out_hv, in_hv], axis=1).fillna(0)
hv_df["hv_total"] = hv_df["hv_in"] + hv_df["hv_out"]





# %% [code] {"execution":{"iopub.status.busy":"2025-12-07T14:55:43.830758Z","iopub.execute_input":"2025-12-07T14:55:43.831092Z","iopub.status.idle":"2025-12-07T14:55:43.853494Z","shell.execute_reply.started":"2025-12-07T14:55:43.831044Z","shell.execute_reply":"2025-12-07T14:55:43.852599Z"}}
btw = nx.betweenness_centrality(subG, normalized=True)
deg_in = dict(subG.in_degree())
deg_out = dict(subG.out_degree())
pagerank = nx.pagerank(subG)


# %% [code] {"execution":{"iopub.status.busy":"2025-12-07T14:56:00.095733Z","iopub.execute_input":"2025-12-07T14:56:00.096173Z","iopub.status.idle":"2025-12-07T14:56:00.105415Z","shell.execute_reply.started":"2025-12-07T14:56:00.096145Z","shell.execute_reply":"2025-12-07T14:56:00.103502Z"}}
cent_df = pd.DataFrame({
    "betweenness": btw,
    "in_degree": pd.Series(deg_in),
    "out_degree": pd.Series(deg_out),
    "pagerank": pagerank
})


# %% [code] {"execution":{"iopub.status.busy":"2025-12-07T14:56:38.730007Z","iopub.execute_input":"2025-12-07T14:56:38.730701Z","iopub.status.idle":"2025-12-07T14:56:38.753411Z","shell.execute_reply.started":"2025-12-07T14:56:38.730683Z","shell.execute_reply":"2025-12-07T14:56:38.751991Z"}}
mule_df = cent_df.join(hv_df).fillna(0)
mule_df.head()

# %% [code] {"execution":{"iopub.status.busy":"2025-12-07T14:58:14.672211Z","iopub.execute_input":"2025-12-07T14:58:14.672618Z","iopub.status.idle":"2025-12-07T14:58:14.699477Z","shell.execute_reply.started":"2025-12-07T14:58:14.672590Z","shell.execute_reply":"2025-12-07T14:58:14.698601Z"}}
# Normalize columns
mule_df["bet_norm"] = mule_df["betweenness"] / mule_df["betweenness"].max()
mule_df["pr_norm"]  = mule_df["pagerank"] / mule_df["pagerank"].max()
mule_df["hv_norm"]  = mule_df["hv_total"] / mule_df["hv_total"].max()
mule_df["deg_norm"] = (mule_df["in_degree"] + mule_df["out_degree"]) / \
                       (mule_df["in_degree"] + mule_df["out_degree"]).max()

# Final mule score
mule_df["mule_score"] = (
    0.40*mule_df["hv_norm"] +
    0.30*mule_df["bet_norm"] +
    0.20*mule_df["pr_norm"] +
    0.10*mule_df["deg_norm"]
)

mule_ranked = mule_df.sort_values("mule_score", ascending=False)
mule_ranked.head(10)


# %% [code] {"execution":{"iopub.status.busy":"2025-12-07T15:32:51.403104Z","iopub.execute_input":"2025-12-07T15:32:51.403834Z","iopub.status.idle":"2025-12-07T15:32:51.459901Z","shell.execute_reply.started":"2025-12-07T15:32:51.403815Z","shell.execute_reply":"2025-12-07T15:32:51.455991Z"}}
import networkx as nx
import matplotlib.pyplot as plt

# ---- Your Inputs ----
G_comm1 = tx_comm1  #subgraph_comm1       # Graph of Community 1 (184 nodes, 284 edges)
centrality = nx.betweenness_centrality(G_comm1)  # or your chosen centrality
txn_amount = nx.get_node_attributes(G_comm1, "TX_AMOUNT")

CENT_THRESH = 0.01
AMOUNT_THRESH = 1000000

# ---- Mule Candidate Detection ----
mule_nodes = [
    n for n in G_comm1.nodes()
    if centrality[n] > CENT_THRESH and txn_amount.get(n, 0) > AMOUNT_THRESH
]

# ---- Node Colors ----
node_colors = [
    "red" if n in mule_nodes else "skyblue"
    for n in G_comm1.nodes()
]

# ---- Node Sizes ----
node_sizes = [
    300 if n in mule_nodes else 80
    for n in G_comm1.nodes()
]

# ---- Plotting ----
plt.figure(figsize=(12, 10))
pos = nx.spring_layout(G_comm1, seed=42, k=0.22)

nx.draw_networkx_edges(G_comm1, pos, alpha=0.3)
nx.draw_networkx_nodes(G_comm1, pos, node_color=node_colors, node_size=node_sizes)

# Label only mule nodes
nx.draw_networkx_labels(
    G_comm1,
    pos,
    labels={n: n for n in mule_nodes},
    font_color="black",
    font_size=8,
    font_weight="bold"
)

plt.title("🔥 Mule Map — High-Risk Mule Candidates in Community 1")
plt.axis("off")
plt.show()


# %% [code] {"execution":{"iopub.status.busy":"2025-12-07T14:34:25.593267Z","iopub.execute_input":"2025-12-07T14:34:25.593634Z","iopub.status.idle":"2025-12-07T14:34:25.606873Z","shell.execute_reply.started":"2025-12-07T14:34:25.593605Z","shell.execute_reply":"2025-12-07T14:34:25.606097Z"}}
nodes_high_value_comm1 = pd.concat([
    high_value_tx_comm1["SENDER_ACCOUNT_ID"],
    high_value_tx_comm1["RECEIVER_ACCOUNT_ID"]
]).unique()

print("Nodes involved (Community 1 high-value):", len(nodes_high_value_comm1))


# %% [code] {"execution":{"iopub.status.busy":"2025-12-07T14:34:55.637733Z","iopub.execute_input":"2025-12-07T14:34:55.638436Z","iopub.status.idle":"2025-12-07T14:34:55.660268Z","shell.execute_reply.started":"2025-12-07T14:34:55.638410Z","shell.execute_reply":"2025-12-07T14:34:55.658748Z"}}
# Outgoing high-value transactions
out_counts = (
    high_value_tx_comm1.groupby("SENDER_ACCOUNT_ID")
    .size()
    .rename("outgoing_high_value_count")
)

# Incoming high-value transactions
in_counts = (
    high_value_tx_comm1.groupby("RECEIVER_ACCOUNT_ID")
    .size()
    .rename("incoming_high_value_count")
)

# Combine
hv_counts_comm1 = pd.concat([out_counts, in_counts], axis=1).fillna(0)
hv_counts_comm1["total_high_value"] = (
    hv_counts_comm1["outgoing_high_value_count"]
    + hv_counts_comm1["incoming_high_value_count"]
)

hv_counts_comm1 = hv_counts_comm1.sort_values("total_high_value", ascending=False)

hv_counts_comm1.head(10)


# %% [code] {"execution":{"iopub.status.busy":"2025-12-07T14:35:26.023306Z","iopub.execute_input":"2025-12-07T14:35:26.023579Z","iopub.status.idle":"2025-12-07T14:35:26.449728Z","shell.execute_reply.started":"2025-12-07T14:35:26.023564Z","shell.execute_reply":"2025-12-07T14:35:26.448431Z"}}
sub_nodes = hv_counts_comm1.index
sub_high_value_G = subG.subgraph(sub_nodes)

pos = nx.spring_layout(sub_high_value_G, seed=42)

plt.figure(figsize=(10, 8))
nx.draw(
    sub_high_value_G,
    pos,
    node_size=120,
    node_color="red",
    edge_color="orange",
    width=2,
    alpha=0.85
)
plt.title("High-Value Actors Inside Community 1", fontsize=14)
plt.show()


KeyboardInterrupt: 